<a href="https://colab.research.google.com/github/parthsharma5575/Ai-Ml-GenAi/blob/main/ParthSharma_Exercises_RAG_Vector_Databases_(14).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Exercises: RAG & Vector Databases

**Post-class practice!** In this notebook, you will build your own mini RAG system step by step.

**Instructions:**
- Complete the lines marked with `# TODO` in each exercise
- Run the code and observe the output carefully
- Answer every "Reflection Question" in the markdown cell provided

**Setup:** Run this on Google Colab (Runtime → Change runtime type → GPU recommended, but CPU works too — just slower)


---
## Part 0: Setup

First, install and import the required libraries.


In [1]:
# Run this first!
!pip install transformers torch sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 72.2 MB/s eta 0:00:00


In [2]:
from transformers import pipeline

# Load the text generation model (same as class)
generator = pipeline('text-generation', model='gpt2')

print("Setup complete! ✅")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Setup complete! ✅


---
## Part 1: Hallucination vs RAG — The "Open Book Exam" 📖

Remember: without context, the LLM **guesses** (Hallucination). With RAG, we give it the "book" to read from.


### Exercise 1.1: Catch the Hallucination! 🕵️

**Task:** Ask GPT-2 a question about something it **cannot possibly know** (a made-up or very recent fact). Observe how it confidently makes something up.

Example ideas: *"Who won the 2027 Cricket World Cup?"*, *"What is the name of the mayor of Atlantis?"*


In [3]:
# TODO: Write a question the model cannot know the answer to
my_question = "Who won 2090 Fifa World Cup?"   # <-- your impossible question here

res = generator(my_question, max_new_tokens=30)
print(res[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=T

Who won 2090 Fifa World Cup? And what about the next FIFA World Cup?

The following 10 questions will be answered in the coming weeks:

Q: What are the


**🤔 Reflection Question 1.1:** Did the model say "I don't know"? Or did it confidently invent an answer? Why do you think LLMs hallucinate instead of admitting they don't know?

> *Your answer here...*


The model did not say "I don't know." Instead, it generated a plausible-sounding response even though the question cannot be answered because the 2090 FIFA World Cup has not happened yet. LLMs hallucinate because they are trained to predict the most likely next words rather than verify whether the information is true or available. Unless they are given external knowledge or instructed to admit uncertainty, they may confidently invent an answer instead of saying they don't know.

### Exercise 1.2: Fix It with RAG (Manual Augmentation) 🔧

**Task:** Now fix the hallucination from Exercise 1.1 using the **Augmentation** step:
1. Write a `private_knowledge` string that contains the correct answer to your question
2. Build a `rag_prompt` that combines **Context + Question** (like we did in class)
3. Compare the output with Exercise 1.1


In [4]:
# TODO: Write the "ground truth" for your question
private_knowledge = "The 2090 Fifa World Cup was won by India"   # <-- e.g., "The 2027 Cricket World Cup was won by ..."

# TODO: Build the RAG prompt — attach context to the question
# Format: Context: ... \n\n Question: ... \n Answer according to the context:
rag_prompt = f""" Context:{private_knowledge}
          Question:{my_question}
          answer according to the context:
 """   # <-- build your prompt here using private_knowledge and my_question

res_rag = generator(rag_prompt, max_new_tokens=50)
print(res_rag[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Context:The 2090 Fifa World Cup was won by India
          Question:Who won 2090 Fifa World Cup?
          answer according to the context:
            answer according to the context:

Question:Why did you choose to host this tournament?

Answer: Because of the economic and strategic benefits of hosting the 2022 World Cup

Question:


**🤔 Reflection Question 1.2:** Which of the 3 RAG steps (Retrieval, Augmentation, Generation) did we do **manually** here? Which step is still missing from our system?

> *Your answer here...*


We manually performed the Augmentation step by adding the correct information (private_knowledge) to the prompt before asking the model the question. The model then performed the Generation step by producing an answer using that context. The missing step is Retrieval, because the relevant information was not automatically searched or fetched from a database or knowledge source—it was manually provided by us.

---
## Part 2: Embeddings — Turning Meaning into Numbers 🔢

Remember: AI understands **Numbers**, not words. Similar meanings → vectors that are **close** on the graph.


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model (same as class)
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready! ✅")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready! ✅


### Exercise 2.1: Look Inside an Embedding 👀

**Task:** Convert a sentence into an embedding and inspect it:
1. Encode the sentence `"I love programming"`
2. Print the **shape** (how many numbers?) and the **first 10 numbers**


In [7]:
sentence = "I love programming"

# TODO: encode the sentence using model.encode()
embedding = model.encode(sentence)   # <-- fix this

print("Shape of embedding:", embedding.shape)
print("First 10 numbers:", embedding[:10])

Shape of embedding: (384,)
First 10 numbers: [-0.03617864 -0.0127737   0.00300628 -0.01690346  0.00948426 -0.06515172
  0.09376637  0.07142349  0.01852629  0.05358268]


**🤔 Reflection Question 2.1:** How many dimensions (numbers) does one embedding have? Can a human read meaning from these numbers directly?

> *Your answer here...*


The embedding has 384 dimensions (384 numbers). Humans cannot directly understand the meaning from these numbers because each value represents a learned feature in a high-dimensional space. Together, the numbers capture the semantic meaning of the sentence, allowing the model to compare and measure similarity between different texts.

### Exercise 2.2: The King & Queen Test 👑

In class we said: *"King" and "Queen" vectors will be close, but "Apple" will be far away.* Let's **prove it with code**!

**Task:** Compute the similarity between sentence pairs. Higher score = closer meaning.


In [8]:
from sentence_transformers import util

sentences = [
    "The king rules the country.",      # 0
    "The queen lives in the palace.",   # 1
    "I ate an apple for breakfast."     # 2
]

embeddings = model.encode(sentences)

# TODO: compute cosine similarity between sentence 0 and sentence 1 (king vs queen)
sim_king_queen = util.cos_sim(embeddings[0], embeddings[1])   # <-- fix the index

# TODO: compute cosine similarity between sentence 0 and sentence 2 (king vs apple)
sim_king_apple = util.cos_sim(embeddings[0], embeddings[2])   # <-- fix the index

print(f"King vs Queen similarity: {sim_king_queen.item():.4f}")
print(f"King vs Apple similarity: {sim_king_apple.item():.4f}")

King vs Queen similarity: 0.4106
King vs Apple similarity: 0.0614


**🤔 Reflection Question 2.2:** Which pair got the higher similarity score? Does this match the "close on the graph" logic from class?

> *Your answer here...*


The King vs Queen pair had the higher similarity score (0.4106), while King vs Apple had a much lower score (0.0614). This matches the "close on the graph" logic because "King" and "Queen" are semantically related, so their embeddings are closer together. "Apple" is unrelated to royalty, so its embedding is much farther away.

---
## Part 3: Build Your Own Vector Database 🗄️

Now the real deal — build a FAISS vector database with **your own documents** and search it.


### Exercise 3.1: Create Your Knowledge Base

**Task:** Write **5 documents of your own** (facts about your college, your city, your favorite topics — anything!). Then:
1. Encode them into vectors
2. Store them in a FAISS index
3. Print how many documents got stored


In [9]:
import faiss

# TODO: Write your own 5 documents (make them about different topics!)
documents = [
    "Bhagwan Parshuram Institute of Technology is located in Rohini, Delhi.",
    "Delhi is the capital city of India and is known for its historical monuments.",
    "Java is an object-oriented programming language widely used for backend development.",
    "Machine learning enables computers to learn patterns from data without explicit programming.",
    "Football is one of the most popular sports in the world and the FIFA World Cup is held every four years."
]

# TODO: Convert documents into vectors using model.encode()
doc_embeddings = model.encode(documents)   # <-- fix this

# TODO: Build the FAISS index
dimension = doc_embeddings.shape[1]        # size of each vector
index = faiss.IndexFlatL2(dimension)                               # <-- create faiss.IndexFlatL2(...) here
index.add(np.array(doc_embeddings,dtype=np.float32))        # add vectors to the database

print(f"{index.ntotal} documents have been stored in the Vector Database.")

5 documents have been stored in the Vector Database.


### Exercise 3.2: Semantic Search Test 🔍

**Task:** Ask a question about one of YOUR documents — but **do NOT use the exact same words** as the document! This tests **Semantic Search** (meaning) vs **Keyword Search** (exact words).

Example: if your document says *"Almonds are healthy for the brain"*, ask *"Which food is good for memory?"*


In [10]:
# TODO: Write a query that matches one document by MEANING, not exact words
query = "Which programming language is commonly used to build backend applications?"   # <-- your query

# TODO: Convert the query into a vector
query_embedding = model.encode([query])   # <-- fix (hint: model.encode([query]))

# TODO: Search the database for the top 1 result
D, I = index.search(np.array(query_embedding,dtype=np.float32), k=1)   # <-- fix k

print(f"Question: {query}")
print(f"Most similar Document: {documents[I[0][0]]}")
print(f"Distance: {D[0][0]:.4f}  (lower = more similar)")

Question: Which programming language is commonly used to build backend applications?
Most similar Document: Java is an object-oriented programming language widely used for backend development.
Distance: 0.7947  (lower = more similar)


**🤔 Reflection Question 3.2:** Did the database find the right document even though you used different words? Explain how this is different from Ctrl+F / keyword search:

> *Your answer here...*


Yes. The vector database found the correct document even though the query used different words. This is because semantic search compares the meaning of the query and the documents using embeddings, rather than looking for exact keyword matches. In contrast, Ctrl+F or keyword search only finds exact words or phrases and may fail if different wording or synonyms are used.

### Exercise 3.3: Top-K Retrieval

**Task:** Instead of only the top 1 result, retrieve the **top 3** most similar documents and print them in order with their distances.


In [11]:
query2 = "Tell me about artificial intelligence and how computers learn from data."   # <-- write another query

query_embedding2 = model.encode([query2])

# TODO: search with k=3
D, I = index.search(np.array(query_embedding2,dtype=np.float32), k=3)   # <-- fix k

print(f"Question: {query2}\n")
# TODO: loop over the 3 results and print rank, document, and distance
for rank in range(3):   # <-- fix range
    print(f"Rank {rank+1}: {documents[I[0][rank]]}  (distance: {D[0][rank]:.4f})")

Question: Tell me about artificial intelligence and how computers learn from data.

Rank 1: Machine learning enables computers to learn patterns from data without explicit programming.  (distance: 0.8082)
Rank 2: Java is an object-oriented programming language widely used for backend development.  (distance: 1.4908)
Rank 3: Bhagwan Parshuram Institute of Technology is located in Rohini, Delhi.  (distance: 1.7025)


**🤔 Reflection Question 3.3:** Look at the distances of ranks 1, 2, and 3. Is the rank-1 document clearly the best match, or are the distances close? When might retrieving more than 1 document be useful for RAG?

> *Your answer here...*


The rank-1 document is clearly the best match because it has the lowest distance (0.8082), while the rank-2 (1.4908) and rank-3 (1.7025) documents are much farther away. Retrieving more than one document is useful in RAG when a question requires information from multiple sources, when the top result is incomplete or ambiguous, or when combining several relevant documents can produce a more accurate and comprehensive answer.

---
## Part 4: 🏆 Final Challenge — Full RAG Pipeline (End-to-End)

Now connect **everything**: Retrieval (Vector DB) → Augmentation (prompt building) → Generation (LLM).

This is the complete flow from class, but built by YOU:

```
User Query → [Vector DB Search] → Best Document → [Attach to Prompt] → [GPT-2] → Answer
```


In [12]:
def my_rag_pipeline(user_query):
    # STEP 1 — RETRIEVAL
    # TODO: encode the user_query into a vector
    q_emb = model.encode([user_query])   # <-- fix

    # TODO: search the FAISS index for the top 1 document
    D, I = index.search(np.array(q_emb,dtype=np.float32), k=1)   # <-- fix
    retrieved_doc = documents[I[0][0]]

    # STEP 2 — AUGMENTATION
    # TODO: build the RAG prompt with the retrieved document as context
    rag_prompt = f"Context:{retrieved_doc}Question:{user_query}Answer according to the context:"
    # <-- Context: ... Question: ... Answer according to the context:

    # STEP 3 — GENERATION
    res = generator(rag_prompt, max_new_tokens=30)

    print("Retrieved Document:", retrieved_doc)
    print("-" * 60)
    print("Final Answer:\n", res[0]['generated_text'])

# Test your pipeline!
my_rag_pipeline("Which programming language is commonly used for backend applications?")   # <-- ask a question about one of your documents

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved Document: Java is an object-oriented programming language widely used for backend development.
------------------------------------------------------------
Final Answer:
 Context:Java is an object-oriented programming language widely used for backend development.Question:Which programming language is commonly used for backend applications?Answer according to the context:Java is an object-oriented programming language widely used for backend development.Question:Which programming language is commonly used for backend applications?Answer according to the


**🤔 Final Reflection Question:**
1. In your pipeline, what would happen if the Vector DB retrieved the **wrong** document? Would the LLM still give a good answer?
2. Based on this, complete the sentence: *"A RAG system is only as good as its ______."*

> *Your answer here...*


If the Vector DB retrieves the wrong document, the LLM will likely generate an incorrect or misleading answer because it relies on the retrieved context. Even a powerful LLM cannot provide a reliable answer if the context it receives is irrelevant or incorrect.

A RAG system is only as good as its retrieval.

### Bonus Challenge 4.1: RAG vs Fine-tuning Decision Table 🎓

For each scenario below, decide: **RAG or Fine-tuning?** (Remember: Fine-tuning = 5 years of med school, RAG = textbook in the exam)

| Scenario | RAG / Fine-tuning? | Reason |
|----------|-------------------|--------|
| A company chatbot that must answer from HR policy PDFs that change every month | Rag | The documents change frequently, so retrieving the latest PDFs is better than retraining the model. |
| Teaching a model to always respond in Shakespearean English style | Fine Tuning | This changes the model's behavior and writing style, which is best learned through fine-tuning. |
| A news assistant that must know today's headlines | Rag | News changes every day, so the model should retrieve the latest information instead of relying on stored knowledge. |
| A medical model that must deeply understand doctor-style reasoning | Fine Tuning | Deep domain expertise and reasoning patterns are learned through fine-tuning on medical data. |

> *Complete the table (edit this cell)...*


### Bonus Challenge 4.2: Break the Retriever! 💥

**Task:** Try to find a query where your Vector DB retrieves the **WRONG** document.

Hints to try:
- A very vague query (e.g., "Tell me something")
- A query about a topic NOT in any of your 5 documents
- A query mixing two topics at once

Print the result and the distance score.


In [13]:
# TODO: try to fool your own retriever!
tricky_query = "Tell me something"   # <-- your tricky query

q_emb = model.encode([tricky_query])
D, I = index.search(np.array(q_emb), k=1)

print(f"Question: {tricky_query}")
print(f"Retrieved: {documents[I[0][0]]}")
print(f"Distance: {D[0][0]:.4f}")

Question: Tell me something
Retrieved: Machine learning enables computers to learn patterns from data without explicit programming.
Distance: 1.6975


**🤔 Reflection Question 4.2:** What happened when you asked about a topic that was NOT in your database? Did the retriever say "not found", or did it still return *something*? Why is this a problem for real RAG systems, and how might we solve it? (Hint: think about the distance score!)

> *Your answer here...*


The retriever did not say "not found." Instead, it still returned the closest document in the database, even though it was not a good match. This is a problem because the LLM may use irrelevant context and generate an incorrect or misleading answer. One way to solve this is to use a distance (or similarity) threshold: if the best match is too far away (high distance/low similarity), the system should return "No relevant document found" instead of passing unrelated context to the LLM.

---
## ✅ Submission Checklist

Before submitting, check:

- [ ] Part 1: Hallucination caught + fixed with manual RAG
- [ ] Part 2: Embedding inspected + King/Queen similarity test done
- [ ] Part 3: Your own 5-document Vector DB built + semantic search + top-3 retrieval
- [ ] Part 4: Full end-to-end RAG pipeline working
- [ ] Bonus 4.1 table filled in
- [ ] All "Reflection Questions" answered
- [ ] **All cells have been run** (outputs are visible)

**Well done! 🎉 You have built a complete RAG system from scratch!**
